# Lab 10.3: Which Training Cases Matter?

Chapter 10. Influence functions, leave-one-out retraining, TracIn and Data Shapley on the Wisconsin breast cancer data bundled with scikit-learn (569 cases, 30 features). Run the cells in order; one optional cell at the end invites you to change a setting.

In [1]:
import numpy as np, warnings, time
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")
Xa, ya = load_breast_cancer(return_X_y=True)          # label 1 = benign, 0 = malignant
rng = np.random.default_rng(0)
idx = rng.permutation(len(ya))
tr, va = idx[:200], idx[200:400]
mu, sd = Xa[tr].mean(0), Xa[tr].std(0)
Xs = (Xa - mu) / sd
Xtr, Xva = np.c_[Xs[tr], np.ones(200)], np.c_[Xs[va], np.ones(200)]
ytr_clean, yva = ya[tr].astype(float), ya[va].astype(float)
ytr = ytr_clean.copy()
flip = rng.choice(200, 20, replace=False)               # plant 20 wrong labels
ytr[flip] = 1 - ytr[flip]
print("training cases:", len(ytr), " flipped:", len(flip), " validation cases:", len(yva))

training cases: 200  flipped: 20  validation cases: 200


**Checkpoint 1:** 200 training cases with 20 flipped labels, 200 validation cases.

## Part A. Influence function versus retraining

We fit an L2-regularized logistic regression with Newton's method, pick one validation case, and estimate for every training case how the loss on that case would change if the training case were removed. Then we check by actually retraining 200 times.

In [2]:
sig = lambda z: 1 / (1 + np.exp(-z))
lam = 0.05
def fit(X, y):
    w = np.zeros(X.shape[1])
    for _ in range(30):
        p = sig(X @ w)
        g = X.T @ (p - y) / len(y) + lam * w
        H = (X.T * (p * (1 - p))) @ X / len(y) + lam * np.eye(X.shape[1])
        w -= np.linalg.solve(H, g)
    return w, H
nll = lambda w, x, t: -(t * np.log(sig(x @ w)) + (1 - t) * np.log(1 - sig(x @ w)))
w, H = fit(Xtr, ytr)
G = (sig(Xtr @ w) - ytr)[:, None] * Xtr                # per-case gradients
losses = nll(w, Xva, yva)
q = int(np.argsort(losses)[-5])                         # a validation case the model finds hard
g_q = (sig(Xva[q] @ w) - yva[q]) * Xva[q]
est = G @ np.linalg.solve(H, g_q) / len(ytr)
t0 = time.time()
loo = np.array([nll(fit(np.delete(Xtr, i, 0), np.delete(ytr, i))[0], Xva[q], yva[q]) for i in range(200)]) - losses[q]
print(f"validation case {q}: loss {losses[q]:.3f}; correlation estimate vs retraining {np.corrcoef(est, loo)[0, 1]:.3f}"
      f"; retraining took {time.time() - t0:.1f} s")
print("top 5 by estimate:", np.argsort(-np.abs(est))[:5], " top 5 by retraining:", np.argsort(-np.abs(loo))[:5])

validation case 77: loss 0.699; correlation estimate vs retraining 0.979; retraining took 0.2 s
top 5 by estimate: [194 116 133 176  68]  top 5 by retraining: [194  23 116 133 176]


**Checkpoint 2:** the correlation between the influence estimate and actual retraining is close to 1 (above 0.95) for this convex model, and the top five lists overlap heavily. Retraining 200 times takes a second or two here; for a large network it would take months.

## Part B. Similar is not influential

In [3]:
d = np.linalg.norm(Xtr[:, :-1] - Xva[q, :-1], axis=1)
nearest = np.argsort(d)[:10]
influential = np.argsort(-np.abs(loo))[:10]
print("10 nearest training cases:", nearest)
print("10 most influential (retraining):", influential)
print("overlap:", len(set(nearest) & set(influential)), "of 10")
print("flipped labels among the 10 most influential:", int(np.isin(influential, flip).sum()))

10 nearest training cases: [ 25  59 102 133  74 117 107  75 146  87]
10 most influential (retraining): [194  23 116 133 176  68 152 146  59  25]
overlap: 4 of 10
flipped labels among the 10 most influential: 6


**Checkpoint 3:** the overlap between the ten nearest and the ten most influential training cases is small. Note how many of the most influential cases carry a flipped label.

## Part C. Finding wrong labels: self-influence, TracIn and Data Shapley

In [4]:
self_if = np.einsum("ij,ij->i", G, np.linalg.solve(H, G.T).T)        # g_i^T H^-1 g_i
wt, lr, tracin = np.zeros(31), 0.5, np.zeros(200)                    # TracIn: gradient descent with checkpoints
for step in range(60):
    Gt = (sig(Xtr @ wt) - ytr)[:, None] * Xtr
    if step % 6 == 0:
        tracin += lr * (Gt ** 2).sum(1)
    wt -= lr * (Gt.mean(0) + lam * wt)

def value(S):                                                        # validation accuracy of a model on subset S
    if len(S) < 5 or len(set(ytr[S])) < 2:
        return 0.5
    return LogisticRegression(max_iter=300).fit(Xtr[S, :-1], ytr[S]).score(Xva[:, :-1], yva)

def data_shapley(T, seed):
    r = np.random.default_rng(seed); full = value(np.arange(200)); phi = np.zeros(200)
    for _ in range(T):
        perm = r.permutation(200); prev = 0.5
        for k in range(1, 201):
            if abs(full - prev) < 0.005 and k > 20:                   # truncation
                break
            cur = value(perm[:k]); phi[perm[k - 1]] += cur - prev; prev = cur
    return phi / T

t0 = time.time()
shap_a = data_shapley(60, seed=1)
print(f"Data Shapley with 60 permutations took {time.time() - t0:.0f} s")
for name, order in (("Data Shapley, lowest first", np.argsort(shap_a)),
                    ("self-influence (IF), highest first", np.argsort(-self_if)),
                    ("TracIn self-influence, highest first", np.argsort(-tracin))):
    print(f"{name:38s}: flipped labels in the first 40 inspected = {int(np.isin(order[:40], flip).sum())} of 20")

Data Shapley with 60 permutations took 9 s
Data Shapley, lowest first            : flipped labels in the first 40 inspected = 17 of 20
self-influence (IF), highest first    : flipped labels in the first 40 inspected = 20 of 20
TracIn self-influence, highest first  : flipped labels in the first 40 inspected = 20 of 20


**Checkpoint 4:** all three methods find most of the 20 flipped labels in the first 40 cases inspected (random inspection would find about 4). In the book's run, the two self-influence scores found all 20 and Data Shapley found 17; your counts may differ by one or two.

## Part D. Is the ranking stable?

In [5]:
shap_b = data_shapley(60, seed=2)
top_a, top_b = set(np.argsort(shap_a)[:20]), set(np.argsort(shap_b)[:20])
print("Data Shapley, two runs with different random permutations:")
print("  correlation of values:", round(np.corrcoef(shap_a, shap_b)[0, 1], 3))
print("  overlap of the 20 lowest-valued cases:", len(top_a & top_b), "of 20")

Data Shapley, two runs with different random permutations:
  correlation of values: 0.649
  overlap of the 20 lowest-valued cases: 15 of 20


**Checkpoint 5:** the two Monte Carlo runs agree on most of the lowest-valued cases but not all, and the correlation of all 200 values is clearly below 1. Monte Carlo noise is a stability problem you must report.

## Questions

1. Part A worked almost perfectly. Name two reasons it would not work as well for a deep network (Chapter 10, Basu et al., 2021).
2. Why does a flipped label have high self-influence? Explain in two sentences without formulas.
3. Data Shapley needed thousands of model fits here, for 200 cases. What would you use for 2 million cases, and what would you lose?
4. Optional: change the number of flipped labels to 40 (cell 1) and rerun. Which method degrades most?